## Instalación de librerías

In [ ]:
!pip install -q scrapy
#!pip install -q newspaper4k
!pip install -q newspaper3k
!pip install -q lxml_html_clean

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.8/259.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 36.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 3.6 MB/s eta 0:00:00


# Articulos de mapas de sitio usando scrapy

In [ ]:
!scrapy startproject news_scraper

New Scrapy project 'news_scraper', using template directory '/usr/local/lib/python3.11/dist-packages/scrapy/templates/project', created in:
    /content/news_scraper

You can start your first spider with:
    cd news_scraper
    scrapy genspider example example.com


## Extracción de URLs

In [1]:
%cd news_scraper

c:\Users\rebec\OneDrive - Universidad de Oviedo\Escritorio\TFG\news_scraper


In [7]:
%%writefile "news_extractor_spider.py"
import scrapy
from newspaper import build, Config, Article
import requests
from urllib.parse import urljoin, urlparse
import os
import time
import re
from datetime import datetime
import gzip
from scrapy.selector import Selector
import dateutil


class NewsUrlExtractorSpider(scrapy.Spider):
    name = 'news_extractor'

    def __init__(self, date="2025-01-01", *args, **kwargs):
        super(NewsUrlExtractorSpider, self).__init__(*args, **kwargs)
        self.from_date = datetime.strptime(date, '%Y-%m-%d').date()
        self.invalid_url_words = {'section', 'tag', 'template',
                                  'category', 'author', 'page-sitemap'
                                  'categories', 'video', 'image', 'temas',
								                  'live', 'microsite', 'focus', 'blog',
                                  'ocio', 'cine', 'board', 'character',
                                  'galeria', 'categoria', 'ficha', 'firmante',
                                  'secciones'}

    # Metadata: nombre y sitemap
    metadata_urls={
        'https://www.elcomercio.es/': {'nombre': 'El Comercio'},
        'https://www.lne.es/': {'nombre': 'La Nueva España'},
        'https://www.lavanguardia.com/': {'nombre': 'La Vanguardia',
                                          'sitemap': 'sitemap-google-news.xml'},
        'https://www.larazon.es/': {'nombre': 'La Razón'},
        'https://www.rtpa.es/': {'nombre': 'Radiotelevisión del Principado de Asturias (RTPA)',
                                 'sitemap': 'sitemap-noticias.xml'},
        'https://www.europapress.es/': {'nombre': 'Europa Press'},
        'https://www.20minutos.es/': {'nombre': '20 Minutos',
                                      'sitemap': 'sitemap-google-news.xml'},
        'https://www.elperiodico.com/' : {'nombre': 'El Periódico',
                                          'sitemap': 'google-news.xml'},
        'https://www.eldiario.es/' : {'nombre': 'ElDiario.es'},
        'https://www.elconfidencial.com/': {'nombre': 'El Confidencial',
                                            'sitemap': 'newsitemap_4.xml'},
        'https://www.culturalgijonesa.org/': {'nombre': 'Cultural Gijonesa'},
        'https://www.elespanol.com/' : {'nombre': 'El Español',
                                        'sitemap': 'sitemap_google_news.xml'},
        'https://www.nortes.me/': {'nombre': 'Nortes'},
        'https://www.lavozdegalicia.es/': {'nombre': 'La Voz de Galicia'},
        'https://www.asturiasmundial.com/': {'nombre': 'Asturias Mundial'},
        'https://www.tribunasalamanca.com/': {'nombre': 'Tribuna Salamanca'},
        'https://migijon.com/': {'nombre': 'Mi Gijón'},
        'https://www.infobae.com/' : {'nombre': 'Infobae',
                                      'sitemap': 'arc/outboundfeeds/news-sitemap2/'},
        'https://www.telecinco.es/' : {'nombre': 'Telecinco'},
        'https://www.laprovincia.es/' : {'nombre': 'La Provincia'},
        'https://www.laopiniondemalaga.es/': {'nombre': 'La Opinión de Málaga'},
        'https://www.elfielato.es/': {'nombre': 'El Fielato y El Nora'},
        'https://www.teleprensa.com/': {'nombre': 'Teleprensa'},

        'https://forbes.es/': {'nombre': 'Forbes'},
        'https://itg.es/': {'nombre': 'ITG centro tecnológico'},
        'https://www.eldia.es/': {'nombre': 'El Dia'},
        'https://www.nationalgeographic.es/': {'nombre': 'Historia National Geographic'},
        'https://iymagazine.es/': {'nombre': 'IyMagazine'},
        'https://www.lacerca.com/': {'nombre': 'La Cerca'},
        'https://www.tribunaavila.com/': {'nombre': 'Tribuna de Ávila'},
        'https://www.latribunadealbacete.es/': {'nombre': 'La Tribuna de Albacete'},
        'https://www.eldiariomontanes.es/': {'nombre': 'El Diario Montañés'},
        "https://www.larioja.com/":{"nombre": "La Rioja"},
        "https://www.almerianoticias.es/":{"nombre": "Almería Noticias"},
        "https://news.ual.es/":{"nombre": "UALNEWS"},
        "https://www.farodevigo.es/":{"nombre": "Faro de Vigo"},
        "https://www.diariocordoba.com/": {'nombre': 'Diario Córdoba'},
        "https://www.diariodeibiza.es/": {'nombre': 'Diario de Ibiza'},
        "https://segoviaudaz.es/": {'nombre': 'SegoviAudaz'},
        'https://www.abc.es/': {'nombre': 'ABC'}, #newspaper
        'https://www.lavozdeasturias.es/': {'nombre': 'La Voz de Asturias'}, #newspaper
        'https://cualia.es/': {'nombre': 'Cualia'}, #newspaper
        'http://www.gentedigital.es/' : {'nombre': 'Gente Digital'}, #newspaper
        'https://www.8directo.com/':{'nombre': '8directo'}, #newspaper
        'https://www.lanuevacronica.com/': {'nombre': 'La Nueva Crónica'}, #newspaper
        'https://elpais.com/': {'nombre': 'El Pais'},
        'https://www.elperiodicodecanarias.es/': {'nombre': 'El Periodico de Canarias'}
    }

    custom_settings = {
        'USER_AGENT': "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:98.0) Gecko/20100101 Firefox/98.0"
    #    'USER_AGENT': "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko; compatible; Googlebot/2.1; +http://www.google.com/bot.html) Chrome/W.X.Y.Z Safari/537.36"
    }

    # Para extracción de xmls
    namespaces = {
        'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9',  # Default namespace
        'news': 'http://www.google.com/schemas/sitemap-news/0.9'
    }

    def get_base_url(self, url):
        parsed_url = urlparse(url)
        return f"{parsed_url.scheme}://{parsed_url.netloc}/"

    def normalize_date(self, date_string):
        try:
            # Conviertir la cadena de fechas en un objeto datetime
            parsed_date = dateutil.parser.parse(date_string)
            # Formatea el objeto datetime como 'YYYY-MM-DD'
            #return parsed_date.strftime('%Y-%m-%d')
            return parsed_date.date()
        except (ValueError, TypeError):
            raise ValueError(f"Formato de fecha inválido: '{date_string}'")

    def is_valid_xml_url(self, sitemap_url):
        for invalid_word in self.invalid_url_words:
            if invalid_word in sitemap_url:
                return False

        pattern = r'\b(1[0-9]{3}|20[0-1][0-9]|202[0-4])\b' # Matches 1000-1999, 2000-2019, and 2020-2024
        match_url = re.search(pattern, sitemap_url)
        if match_url:
            return False
        else:
            return True

    def start_requests(self):
        for url in self.metadata_urls:
            yield scrapy.Request(url=url, callback=self.parse)

    def parse(self, response):
        robots_url = urljoin(response.url, "/robots.txt")
        print(f'Robots URL: {robots_url}')
        yield scrapy.Request(robots_url, callback=self.parse_robots, meta={'domain': response.url})


    def parse_robots(self, response):
        if response.status != 200:
            print(f"Error al acceder a la url: {response.status}")
            return

        domain = response.meta['domain']
        domain_base = self.get_base_url(domain)
        current_sitemap_urls = set() #[]

        # Extraer urls de sitemaps de robots.txt
        for line in response.text.splitlines():
            if line.lower().startswith('sitemap:'):
                sitemap_url = line.split(':', 1)[1].strip()
                parsed_url = urlparse(sitemap_url)
                file_path = parsed_url.path
                file_name, file_extension = os.path.splitext(file_path)
                #if file_extension.lower() == ".xml" and self.is_valid_xml_url(sitemap_url):
                if self.is_valid_xml_url(sitemap_url):
                    current_sitemap_urls.add(sitemap_url)

        if len(current_sitemap_urls) > 0:
            print(f"Se encontraron los siguientes enlaces xml: {current_sitemap_urls}")
            for sitemap_url in current_sitemap_urls:
                print(f"Accediendo al siguiente enlace... {sitemap_url}")
                #yield scrapy.Request(sitemap_url, callback=self.parse_sitemap)  # Llamada recursiva
                yield scrapy.Request(sitemap_url, callback=lambda response: self.parse_sitemap(response, domain) )

        elif domain_base in self.metadata_urls and 'sitemap' in self.metadata_urls[domain_base]:
            print("No se encontraron enlaces xml en robots.txt.", end=" ")
            sitemap_name = self.metadata_urls[domain_base]['sitemap']
            sitemap_url = urljoin(domain_base, sitemap_name)
            print(f"Accediendo al siguiente enlace especificado... {sitemap_url}")
            yield scrapy.Request(sitemap_url, callback=lambda response: self.parse_sitemap(response, domain_base) )
        else:
            print("No se encontraron enlaces xml. Accediendo a enlaces con librería Newspaper...")
            # Si no se encuentra ningún mapa del sitio,
            # utilizar Newspaper4k para obtener las URL de los artículos de noticias
            yield from self.get_news_urls(domain)



    def parse_sitemap(self, response, domain=None):

        # Sitemaps anidados. Tag <sitemap>
        for sitemap in response.xpath('//ns:sitemap', namespaces=self.namespaces):
            sitemap_loc = sitemap.xpath('./ns:loc/text()', namespaces=self.namespaces).get()

            # Omitir sitemaps antiguos
            last_mod = sitemap.xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()
            if last_mod:
                lastmod = self.normalize_date(last_mod)
                if lastmod < self.from_date:
                    continue

            # Verificar que la url es un archivo xml
            parsed_url = urlparse(sitemap_loc)
            file_path = parsed_url.path
            file_name, file_extension = os.path.splitext(file_path)

            #if file_extension.lower() == ".xml" and self.is_valid_xml_url(sitemap_loc):#'section' not in sitemap_loc:
            if file_extension.lower() != '.gz' and self.is_valid_xml_url(sitemap_loc):#'section' not in sitemap_loc:
                #print(f'Se encontró otro xml dentro del archivo actual: {sitemap_loc}') ## lots of outputs
                if sitemap_loc:
                    #yield scrapy.Request(sitemap_loc, callback=self.parse_sitemap)  # Llamada recursiva
                    yield scrapy.Request(sitemap_loc, callback=lambda response: self.parse_sitemap(response, domain) )
            elif file_extension.lower() == '.gz'  and self.is_valid_xml_url(sitemap_loc):#'section' not in sitemap_loc:
                print(f"Se encontró archivo comprimido {sitemap_loc}")
                #yield scrapy.Request(sitemap_loc, callback=self.parse_sitemap_gz)
                yield scrapy.Request(sitemap_loc, callback=lambda response: self.parse_sitemap_gz(response, domain) )



        # Verificar que el archivo tenga noticias recientes
        selector_list = response.xpath('//ns:url', namespaces=self.namespaces)
        if len(selector_list) > 1:
            first_publication_date = selector_list[0].xpath('./news:news/news:publication_date/text()', namespaces=self.namespaces).get()
            if first_publication_date is None:
                first_publication_date = selector_list[0].xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()

            last_publication_date = selector_list[-1].xpath('./news:news/news:publication_date/text()', namespaces=self.namespaces).get()
            if last_publication_date is None:
                last_publication_date = selector_list[-1].xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()

            first_publication_date = self.normalize_date(first_publication_date)
            last_publication_date = self.normalize_date(last_publication_date)
            if first_publication_date < self.from_date and last_publication_date < self.from_date:
                return

        # Extraer datos de cada tag <url>
        for url in selector_list:
        #for url in response.xpath('//ns:url', namespaces=self.namespaces):
            loc = url.xpath('./ns:loc/text()', namespaces=self.namespaces).get()
            title = url.xpath('./news:news/news:title/text()', namespaces=self.namespaces).get()
            publication_date = url.xpath('./news:news/news:publication_date/text()', namespaces=self.namespaces).get()
            fuente = url.xpath('./news:news/news:publication/news:name/text()', namespaces=self.namespaces).get()

            if loc is None:
                break

            title = "" if title is None else title

            if publication_date is None:
                publication_date = url.xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()

            if publication_date is not None:
                publication_date_comp = self.normalize_date(publication_date)
                #publication_date_ = dateutil.parser.parse(publication_date).date()
                if publication_date_comp < self.from_date:
                    continue
                    #break
            else:
                continue


            if domain is not None:
                domain_base = self.get_base_url(domain)
                if domain_base in self.metadata_urls:
                    fuente = self.metadata_urls[domain_base]['nombre']

            yield {
                'fuente': fuente,
                'url': loc,
                'titulo': title,
                'fecha_publicacion': publication_date,
            }

    def parse_sitemap_gz(self, response, domain=None):

        compressed_file = response.body
        decompressed_file = gzip.decompress(compressed_file).decode("utf-8")
        selector = Selector(text=decompressed_file, type="xml")
        #yield from self.parse_sitemap(selector)
        yield from self.parse_sitemap(selector, domain)

    def get_news_urls(self, domain):

        config = Config()
        config.request_timeout = 5
        config.language= 'es'
        config.thread_timeout_seconds = 5
        config.memoize_articles = False
        config.fetch_images = False
        config.follow_meta_refresh = True
        config.number_threads = 4
        config.browser_user_agent = self.custom_settings['USER_AGENT']
        config.headers = {
            "User-Agent": self.custom_settings['USER_AGENT'],
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Accept-Encoding": "gzip, deflate",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "none",
            "Sec-Fetch-User": "?1",
            "Cache-Control": "max-age=0",
        }

        try:
            print(f'Extrayendo noticias con librería Newspaper para url: {domain}')
            start_time = time.time()
            paper = build(domain, config=config)
            print(f"--- {(time.time() - start_time):.2f}s segundos ---")
            articulos_urls = set(paper.article_urls())
            print(f'Se encontraron {len(articulos_urls)} enlaces de noticias.')

            # Obtener fuente
            if domain is not None:
                domain_base = self.get_base_url(domain)
                if domain_base in self.metadata_urls:
                    fuente = self.metadata_urls[domain_base]['nombre']
                else:
                    found_match = re.search(r"www\.(.*?)\.(.+)", domain)
                    fuente = ''
                    if found_match:
                        fuente = found_match.group(1)


            titulo, fecha_publicacion = '', None
            for articulo_url in articulos_urls:
                yield {
                    'fuente': fuente,
                    'url': articulo_url,
                    'titulo': titulo,
                    'fecha_publicacion': fecha_publicacion,
                }
        except Exception as e:
            print(f"No se pudieron obtener noticias de {domain}")

Overwriting news_extractor_spider.py


Copiar archivo spider a carpeta para ejecutar

In [12]:
# !cp news_extractor_spider.py news_scraper/spiders
!copy news_extractor_spider.py news_scraper\spiders


        1 archivo(s) copiado(s).


Eliminar csv para generar uno nuevo

In [13]:
# Eliminar csv
import os
output_file_csv = "news_output.csv"
if os.path.exists(output_file_csv):
    os.remove(output_file_csv)

Ejecutar spyder de scrapy para gener el archivo csv con los enlaces

In [14]:
!scrapy crawl news_extractor -o news_output.csv -s LOG_ENABLED=False -s ROBOTSTXT_OBEY=False -a date=2025-02-15
#!scrapy crawl news_extractor -o news_output.csv -s ROBOTSTXT_OBEY=False

Robots URL: https://www.elcomercio.es/robots.txt
Robots URL: https://www.20minutos.es/robots.txt
Robots URL: https://www.europapress.es/robots.txt
Robots URL: https://www.lne.es/robots.txt
Robots URL: https://www.elconfidencial.com/robots.txt
Robots URL: https://www.larazon.es/robots.txt
Robots URL: https://www.eldiario.es/robots.txt
Robots URL: https://www.asturiasmundial.com/robots.txt
Se encontraron los siguientes enlaces xml: {'https://www.elcomercio.es/sitemap.incremental.xml', 'https://www.elcomercio.es/sitemap.xml'}
Accediendo al siguiente enlace... https://www.elcomercio.es/sitemap.incremental.xml
Accediendo al siguiente enlace... https://www.elcomercio.es/sitemap.xml
Robots URL: https://www.rtpa.es/robots.txt
Robots URL: https://www.culturalgijonesa.org/robots.txt
Robots URL: https://migijon.com/robots.txt
Se encontraron los siguientes enlaces xml: {'https://www.lne.es/sitemap_today_52e19.xml', 'https://www.lne.es/sitemap_google_news_52e19.xml'}
Accediendo al siguiente enlace.

## Filtrar articulos por fecha

In [20]:
import pandas as pd
from datetime import datetime
import numpy as np

# Cargar la data extraída
data_df = pd.read_csv("news_output.csv")
# Fecha inicial
fecha_inicial = "2025-2-21"

data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'], format='mixed', errors='coerce', utc=True)
data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'].dt.strftime("%Y-%m-%d"))
data_df = data_df[(data_df['fecha_publicacion'].between(fecha_inicial, datetime.now().strftime('%Y-%m-%d'))) | (pd.isna(data_df['fecha_publicacion']))]
data_df = data_df.sort_values(by='fecha_publicacion', ascending=False).drop_duplicates().reset_index(drop=True)
data_df

,fuente,url,titulo,fecha_publicacion
0,La Nueva España,https://www.lne.es/oriente/2025/03/02/rey-carn...,"El ""rey"" del Carnaval llega desde Gijón y se c...",2025-03-02
1,Faro de Vigo,https://www.farodevigo.es/motogp/2025/03/02/ma...,Marc Márquez gana en Tailandia y completa un d...,2025-03-02
2,Diario Córdoba,https://www.diariocordoba.com/tiempo/2025/03/0...,TIEMPO CARMONA | El tiempo en Carmona: previsi...,2025-03-02
3,Diario Córdoba,https://www.diariocordoba.com/tiempo/2025/03/0...,TIEMPO CAMAS | El tiempo en Camas: previsión m...,2025-03-02
4,Diario Córdoba,https://www.diariocordoba.com/tiempo/2025/03/0...,TIEMPO CABRA | El tiempo en Cabra: previsión m...,2025-03-02
...,...,...,...,...
58216,El Periodico de Canarias,https://www.elperiodicodecanarias.es/los-llano...,NaN,NaT
58217,El Periodico de Canarias,https://www.elperiodicodecanarias.es/gamarra-d...,NaN,NaT
58218,El Periodico de Canarias,https://www.elperiodicodecanarias.es/ucrania-o...,NaN,NaT
58219,El Periodico de Canarias,https://www.elperiodicodecanarias.es/forjando-...,NaN,NaT


In [21]:
data_df['fuente'].value_counts()

fuente
Europa Press                                         15526
Faro de Vigo                                          5392
La Razón                                              4587
Teleprensa                                            4555
El Dia                                                3713
La Opinión de Málaga                                  3712
Diario de Ibiza                                       3386
ElDiario.es                                           3071
El Comercio                                           1439
El Periódico                                          1437
Forbes                                                1197
La Cerca                                              1155
Diario Córdoba                                         981
La Rioja                                               979
El Diario Montañés                                     838
La Provincia                                           799
La Vanguardia                                    

### Eliminar enlaces duplicados

In [22]:
import os
def remover_url_duplicados(df):
    df['non_null_count'] = df.notnull().sum(axis=1)
    df = df.sort_values(by=['url', 'non_null_count'], ascending=[True, False])
    df = df.drop_duplicates(subset=['url'], keep='first')
    df = df.drop(columns=['non_null_count'])
    df = df.sort_values(by='fecha_publicacion', ascending=False).reset_index(drop=True)
    return df

data_df = remover_url_duplicados(data_df)
data_df

# Obtener la fecha actual en formato YYYY-MM-DD
fecha_hoy = datetime.today().strftime('%Y-%m-%d')

# Definir la ruta del directorio
output_dir = 'news_scraper'
os.makedirs(output_dir, exist_ok=True)

# Guardar en un archivo CSV dentro de la carpeta new_scraper
data_df.to_csv(os.path.join(output_dir, f'{fecha_hoy}.csv'), index=False)

data_df

,fuente,url,titulo,fecha_publicacion
0,Tribuna Salamanca,https://www.tribunasalamanca.com/noticias/3959...,Persecución y tiroteo en las calles de Salaman...,2025-03-02
1,La Vanguardia,https://www.lavanguardia.com/internacional/202...,"Palermo, la Cosa Nostra sigue aquí",2025-03-02
2,El Dia,https://www.eldia.es/opinion/2025/03/02/manos-...,En manos de quién estamos,2025-03-02
3,La Vanguardia,https://www.lavanguardia.com/internacional/202...,“Hemos de seguir en el frente”,2025-03-02
4,El Dia,https://www.eldia.es/opinion/2025/03/02/muerto...,OPINION: Muertos de mierda,2025-03-02
...,...,...,...,...
48311,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/asturia...,NaN,NaT
48312,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/internac...,NaN,NaT
48313,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/sociedad...,NaN,NaT
48314,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/vigo/202...,NaN,NaT


In [23]:
data_df['fuente'].value_counts()

fuente
Europa Press                                         10538
Faro de Vigo                                          4887
Teleprensa                                            4555
La Razón                                              3873
El Dia                                                3552
La Opinión de Málaga                                  3309
Diario de Ibiza                                       3004
ElDiario.es                                           2632
Forbes                                                1197
La Cerca                                              1155
El Comercio                                           1121
El Periódico                                          1014
La Vanguardia                                          774
La Rioja                                               765
El Confidencial                                        680
El Diario Montañés                                     639
El Español                                       

In [24]:
import pandas as pd
from datetime import datetime, timedelta

# Cargar la data extraída
data_df = pd.read_csv("news_output.csv")

# Obtener la fecha de ayer y hoy
hoy = datetime.utcnow().strftime('%Y-%m-%d')
ayer = (datetime.utcnow() - timedelta(days=1)).date()

# Convertir las fechas de publicación
data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'], format='mixed', errors='coerce', utc=True)

# Convertir a solo fecha (sin hora)
data_df['fecha_publicacion'] = data_df['fecha_publicacion'].dt.date

# Filtrar solo las noticias de ayer y hoy
data_df = data_df[data_df['fecha_publicacion'].isin([ayer, datetime.utcnow().date()])]

# Eliminar duplicados y ordenar por fecha de publicación
data_df = data_df.sort_values(by='fecha_publicacion', ascending=False).drop_duplicates().reset_index(drop=True)


# Guardar el CSV con la fecha de hoy en el nombre
nombre_archivo = f"{hoy}.csv"
data_df.to_csv(nombre_archivo, index=False)

# Mostrar el nombre del archivo guardado
print(f"Archivo guardado como: {nombre_archivo}")

# Mostrar el resultado
data_df


Archivo guardado como: 2025-03-02.csv


,fuente,url,titulo,fecha_publicacion
0,La Nueva España,https://www.lne.es/oriente/2025/03/02/rey-carn...,"El ""rey"" del Carnaval llega desde Gijón y se c...",2025-03-02
1,Faro de Vigo,https://www.farodevigo.es/celta-de-vigo/2025/0...,La Ponferradina pone a prueba al Celta Fortuna,2025-03-02
2,Faro de Vigo,https://www.farodevigo.es/pontevedra/2025/03/0...,La Brilat se adiestra en el hielo de Eslovaquia,2025-03-02
3,Europa Press,https://www.europapress.es/la-rioja/noticia-ma...,Marqués de Murrieta preservará el ecosistema ú...,2025-03-02
4,Europa Press,https://www.europapress.es/internacional/notic...,Alemania calibra el impacto de las elecciones ...,2025-03-02
...,...,...,...,...
12389,Europa Press,https://www.europapress.es/illes-balears/notic...,A juicio un hombre por abusar sexualmente de u...,2025-03-01
12390,Europa Press,https://www.europapress.es/galicia/noticia-ind...,"La industria automovilística gallega ve con ""p...",2025-03-01
12391,Europa Press,https://www.europapress.es/la-rioja/noticia-ma...,"Más de 300 corredores, este domingo en la 18 e...",2025-03-01
12392,Europa Press,https://www.europapress.es/deportes/motociclis...,Marc Márquez arrasa con Ducati y se impone en ...,2025-03-01


## Acceder al contenido de las noticias

### Extracción con request

In [ ]:
import requests
from bs4 import BeautifulSoup


# Lista de user-agents para probar
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
    "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko; compatible; Googlebot/2.1; +http://www.google.com/bot.html) Chrome/W.X.Y.Z Safari/537.36"
]

def extraer_texto_articulos_request(url):
    try:
        # Primer intento sin user-agent
        response = requests.get(url)

        # Si la respuesta no es exitosa o el contenido está vacío, intente con agentes de usuario
        if response.status_code != 200 or not response.content:
            for user_agent in user_agents:
                headers = {
                    "User-Agent": user_agent,
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
                    "Accept-Language": "en-US,en;q=0.5",
                    "Accept-Encoding": "gzip, deflate",
                    "Connection": "keep-alive",
                }
                response = requests.get(url, headers=headers)

                if response.status_code == 200 and response.content:
                    #print("Encontrado con user-agent")
                    break

        html_str = response.content
        soup = BeautifulSoup(html_str, 'lxml')

        # Remover footer
        footer = soup.find('footer')
        if footer:
            footer.decompose()

        text_content = soup.find_all('p')

        text = ''
        for i in text_content:
            text += i.text.strip() + " "

        return text.strip() if text.strip() else None

    except Exception as e:
        print(f"Error in url: {url}, error: {e}")
        return None

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

num_threads = 16

start_time = time.time()
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    data_df['texto'] = list(executor.map(extraer_texto_articulos_request, data_df['url']))
print(f"--- {(time.time() - start_time):.2f}s seconds ---")

<ipython-input-376-3475ade41072>:35: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_str, 'lxml')


--- 1999.83s seconds ---


In [ ]:
data_df

,fuente,url,titulo,fecha_publicacion,texto
0,Infobae,https://www.infobae.com/america/agencias/2024/...,"Eddy Merckx, tras romperse la cadera en bici: ...",2025-02-23,"23 Feb, 2025 Por Newsroom Infobae Bruselas, ..."
1,La Nueva España,https://www.lne.es/economia/2025/02/23/marufin...,"Marufina, los armadores que dejaron el mar par...",2025-02-23,EMPRESAS Rafael López muestra uno de los carpa...
2,Europa Press,https://www.europapress.es/murcia/noticia-prev...,"Previsión meteorológica para este domingo, 23 ...",2025-02-23,Menú Boletines Abonados MURCIA 23 Feb. (EUROPA...
3,El Comercio,https://www.elcomercio.es/cronica-negra/rastro...,El rastro sangriento del asesino en serie de T...,2025-02-23,MiComercio Mis noticias Mis intereses Newslett...
4,La Opinión de Málaga,https://www.laopiniondemalaga.es/sociedad/2025...,"El Papa pasa una noche ""tranquila"" después de ...",2025-02-23,Vaticano El papa Francisco. / EP EP El Vatica...
...,...,...,...,...,...
17091,ABC,https://www.semana.es/familia-real-espanola/de...,NaN,NaT,Ingrediente antiedad Carolina de Mónaco Marido...
17092,ABC,https://www.semana.es/familia-real-espanola/pr...,NaN,NaT,Reina Letizia Ingrediente antiedad Looks Fashi...
17093,ABC,https://www.semana.es/living/entramos-casa-osc...,NaN,NaT,Reina Letizia Ingrediente antiedad Looks Fashi...
17094,ABC,https://www.semana.es/moda/nagore-robles-a-ani...,NaN,NaT,Reina Letizia Ingrediente antiedad Looks Fashi...


In [ ]:
data_df['texto'].isna().sum()

1435

In [ ]:
(data_df['texto'] == '').sum()

0

In [ ]:
data_df['texto'].iloc[-2]

'Reina Letizia Ingrediente antiedad Looks Fashion Week Marido Laura Madrueño  Carolina de Mónaco Iker Casillasa Pilar Molina Letizia Supervivientes Amalia de Holanda Itziar Miranda Horóscopo diario: Hoy, domingo 23 de febrero de 2025 Ya en tu quiosco Moda Redactor especializado en Moda y Belleza Actualizado a 21 de febrero de 2025, 11:18 De Nagore Robles a Anita Matamoros: los peores y mejores looks de las influencers en la primera jornada de la Madrid Fashion Week 2025 Ayer dio comienzo la Mercedes Benz Fashion Week Madrid 2025, y como cada año, las expectativas estaban por las nubes. La cosa empezó fuerte con un desfile de Baro Lucas, una auténtica declaración de intenciones que marcó el tono de lo que sería una jornada llena de talento y espectáculo. El diseñador nos dejó boquiabiertos con su capacidad de reinventar siluetas clásicas en clave contemporánea. Pero esto solo fue el aperitivo. La jornada continuó con desfiles de diseñadores de renombre: Pedro del Hierro, Álex Riviere, M

In [ ]:
data_df.iloc[-3].values

array(['ABC',
       'https://www.semana.es/living/entramos-casa-oscar-higares-su-refugio-espacios-abiertos-salon-a-doble-altura-impresionante-jardin-y-museo-aire-libre_2794918',
       nan, NaT,
       'Reina Letizia Ingrediente antiedad Looks Fashion Week Marido Laura Madrueño  Carolina de Mónaco Iker Casillasa Pilar Molina Letizia Supervivientes Amalia de Holanda Itziar Miranda Horóscopo diario: Hoy, domingo 23 de febrero de 2025 Ya en tu quiosco Living Periodista especializada en Corazón Actualizado a 20 de febrero de 2025, 21:11 Óscar Higares en su casa de Madrid Su casa es ese refugio al que siempre volver y donde pasa la mayor parte del tiempo. Pese a tener que viajar, en numerosas ocasiones, por trabajo, Óscar Higares siempre regresa a casa. "Aunque parezca lo contrario, paso la mayor parte del tiempo en mi casa. No suelo ir a sitios donde me tenga que quedar a dormir. No viajaba a nada que me impidieran luego volver a casa", confesaba él mismo hace unos años en una entrevista.

## Filtrar textos

### Buscar textos que tengan al menos una palabra clave

In [ ]:
# Filtrar por palabras clave
keywords = ['universidad', 'oviedo']

# Regex pattern
pattern = '|'.join(keywords)

# Filtrar dataframe
filtered_df = data_df[data_df['texto'].str.contains(pattern, case=False, na=False)].reset_index(drop=True)

filtered_df

,fuente,url,titulo,fecha_publicacion,texto
0,La Opinión de Málaga,https://www.laopiniondemalaga.es/sociedad/2025...,Más parcialidad y menos sueldo: al ritmo actua...,2025-02-23,DESIGUALDAD Las mujeres trabajan gratis varios...
1,La Vanguardia,https://www.lavanguardia.com/cultura/culturas/...,Más divertido que ver cómo se seca la pintura,2025-02-23,Cultura|s Una escena de 'Paint Draying' Begoña...
2,La Nueva España,https://www.lne.es/deportes/deporte-astur/2025...,El Vetusta reina en el derbi ovetense del grup...,2025-02-23,El jugador del Vetusta Carbajo lanza a la port...
3,La Nueva España,https://www.lne.es/deportes/deporte-astur/2025...,El Universidad remonta en Ávila (89-94) y se a...,2025-02-23,Gran triunfo en el que consiguió anoche el Uni...
4,La Nueva España,https://www.lne.es/deportes/deporte-astur/2025...,El Club Bádminton Oviedo gana al Alicante y ju...,2025-02-23,"Por la izquierda, Nicolás García, Yelozabeta, ..."
...,...,...,...,...,...
2902,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/yes/2025...,NaN,NaT,"Año 2024, primero de bachillerato en una clase..."
2903,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/yes/2025...,NaN,NaT,Luz llevaba 13 años de relación con su anterio...
2904,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/yes/2025...,NaN,NaT,Un ejemplo para los que se sorprenden por la e...
2905,La Voz de Galicia,https://www.lavozdegalicia.es/xlsemanal/cienci...,NaN,NaT,Récord submarino https://www.lavozdegalicia.es...


In [ ]:
filtered_df['url'].values[0:100]

array(['https://www.laopiniondemalaga.es/sociedad/2025/02/23/parcialidad-sueldo-ritmo-actual-espana-114594185.html',
       'https://www.lavanguardia.com/cultura/culturas/20250223/10393586/mas-divertido-ver-como-seca-pintura.html',
       'https://www.lne.es/deportes/deporte-astur/2025/02/23/vetusta-reina-derbi-ovetense-grupo-114591063.html',
       'https://www.lne.es/deportes/deporte-astur/2025/02/23/universidad-remonta-avila-89-94-114591071.html',
       'https://www.lne.es/deportes/deporte-astur/2025/02/23/club-badminton-oviedo-gana-alicante-114591072.html',
       'https://www.lne.es/deportes/deporte-astur/2025/02/23/lobas-le-sobra-ultima-jugada-114591070.html',
       'https://www.lne.es/deportes/deporte-astur/2025/02/23/lucas-langarita-experiencia-seleccion-espanola-114591420.html',
       'https://www.lne.es/gijon/2025/02/23/proxima-semana-negra-memoria-angel-114591417.html',
       'https://www.lne.es/gijon/2025/02/23/abraham-menendez-ilustrador-estrellas-cine-114592935.html',

## Buscar textos que tengan todas las palabras clave

In [ ]:
# Lista de palabras clave
keywords = ['universidad', 'oviedo']

# Crear una condición de filtro para cada palabra clave
filter_condition = data_df['texto'].str.contains(keywords[0], case=False, na=False)

for keyword in keywords[1:]:
    filter_condition &= data_df['texto'].str.contains(keyword, case=False, na=False)

# Aplicar filtro
filtered_df_all = data_df[filter_condition].reset_index(drop=True)

filtered_df_all

,fuente,url,titulo,fecha_publicacion,texto
0,La Opinión de Málaga,https://www.laopiniondemalaga.es/sociedad/2025...,Más parcialidad y menos sueldo: al ritmo actua...,2025-02-23,DESIGUALDAD Las mujeres trabajan gratis varios...
1,La Nueva España,https://www.lne.es/deportes/deporte-astur/2025...,El Universidad remonta en Ávila (89-94) y se a...,2025-02-23,Gran triunfo en el que consiguió anoche el Uni...
2,La Nueva España,https://www.lne.es/gijon/2025/02/23/abraham-me...,"Abraham Menéndez, el ilustrador de las estrell...",2025-02-23,La figura de la semana El ilustrador de las es...
3,La Nueva España,https://www.lne.es/asturias/2025/02/23/guia-pe...,POLÉMICAS PARQUES DE BATERÍAS | Guía para no p...,2025-02-23,Parques de baterías en Asturias Xuan Fernández...
4,La Nueva España,https://www.lne.es/asturias/2025/02/23/peticio...,LEY DE CIENCIA ASTURIAS | Las peticiones de lo...,2025-02-23,alarmLas peticiones de los científicos asturia...
...,...,...,...,...,...
129,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/oviedo/...,NaN,NaT,La Consejería de Educación inaugurará el próxi...
130,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/oviedo/...,NaN,NaT,Andrew Silin y Kate Yakusheva se conocían de c...
131,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/siero/2...,NaN,NaT,"El alcalde de Siero, Ángel García, se ha mostr..."
132,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/turismo...,NaN,NaT,La Comarca Vaqueira toma su nombre de los vaqu...


In [ ]:
def mostrar_informacion(data_df):

    for index, fila in data_df.iterrows():
        if pd.isna(fila['fecha_publicacion']):
            fecha = 'Desconocido'
        else:
            fecha = fila['fecha_publicacion'].strftime('%Y/%m/%d')
        fuente = fila['fuente']

        titulo = 'Desconocido'
        if pd.notna(fila['titulo']):
            titulo = fila['titulo']

        print(titulo)
        print(fila['url'])
        print(f'{fecha} - {fuente}')
        print()

mostrar_informacion(filtered_df_all)

Más parcialidad y menos sueldo: al ritmo actual, España tardará 32 años en cerrar la brecha salarial de género
https://www.laopiniondemalaga.es/sociedad/2025/02/23/parcialidad-sueldo-ritmo-actual-espana-114594185.html
2025/02/23 - La Opinión de Málaga

El Universidad remonta en Ávila (89-94) y se acerca al segundo puesto
https://www.lne.es/deportes/deporte-astur/2025/02/23/universidad-remonta-avila-89-94-114591071.html
2025/02/23 - La Nueva España

Abraham Menéndez, el ilustrador de las estrellas de cine que adora pasear por el Muro
https://www.lne.es/gijon/2025/02/23/abraham-menendez-ilustrador-estrellas-cine-114592935.html
2025/02/23 - La Nueva España

POLÉMICAS PARQUES DE BATERÍAS | Guía para no perderse en la polémica de los parques de baterías en Asturias: qué son y por qué no se para de hablar de ellos
https://www.lne.es/asturias/2025/02/23/guia-perderse-polemica-parques-baterias-114588061.html
2025/02/23 - La Nueva España

LEY DE CIENCIA ASTURIAS | Las peticiones de los científi